# 03 — EDA, Diagnostics and Regime Analysis

## Purpose

This notebook studies the statistical behaviour of the **development dataset only**.

It answers:

- Are TT levels stationary?
- Are changes and returns stationary?
- Is there serial dependence?
- Are there ARCH / volatility-clustering effects?
- Are there structural breaks?
- How do currencies move together over time?
- Are there distinct low/high-volatility regimes?
- Which target formulation is more suitable for later modelling?

## Important rule

The final 2024–2025 holdout is **not loaded or evaluated here**.

In [ ]:
# ============================================================
# 1. IMPORTS + CONFIGURATION
# ============================================================

from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy import stats

from statsmodels.tsa.stattools import (
    adfuller,
    kpss,
    acf,
    pacf,
    zivot_andrews,
)
from statsmodels.stats.diagnostic import (
    acorr_ljungbox,
    het_arch,
)
from statsmodels.stats.diagnostic import breaks_cusumolsresid
import statsmodels.api as sm

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 150)
pd.set_option("display.width", 180)
pd.set_option("display.float_format", lambda x: f"{x:,.6f}")

CURRENCIES = ["USD", "GBP", "EUR", "JPY"]
HORIZONS = [1, 3, 7]

print("Diagnostics environment ready.")

In [ ]:
# ============================================================
# 2. LOCATE PROJECT + LOAD DEVELOPMENT DATA ONLY
# ============================================================

def find_project_root():
    candidates = [
        Path.cwd(),
        Path.cwd().parent,
        Path.cwd().parent.parent,
    ]

    for root in candidates:
        if (root / "data" / "processed" / "02_development_model_dataset.csv").exists():
            return root.resolve()

    raise FileNotFoundError(
        "Could not find data/processed/02_development_model_dataset.csv. "
        "Run Script 02 first."
    )

PROJECT_ROOT = find_project_root()

DATA_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "02_development_model_dataset.csv"
)

TABLE_DIR = PROJECT_ROOT / "outputs" / "tables"
FIGURE_DIR = PROJECT_ROOT / "outputs" / "figures"

TABLE_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(DATA_PATH, parse_dates=["Date"])

for h in HORIZONS:
    c = f"Target_Date_H{h}"
    if c in df.columns:
        df[c] = pd.to_datetime(df[c], errors="coerce")

df = df.sort_values("Date").reset_index(drop=True)

print("Loaded development data:", df.shape)
print("Date range:", df["Date"].min().date(), "to", df["Date"].max().date())

assert df["Date"].max() < pd.Timestamp("2024-01-01")
print("PASS — final holdout is not present.")

# 3. Basic development-period overview

In [ ]:
overview_rows = []

for cur in CURRENCIES:
    s = pd.to_numeric(df[f"{cur}_mid"], errors="coerce")

    overview_rows.append({
        "Currency": cur,
        "Observations": int(s.notna().sum()),
        "Start_Level": s.dropna().iloc[0],
        "End_Level": s.dropna().iloc[-1],
        "Min_Level": s.min(),
        "Max_Level": s.max(),
        "Mean_Level": s.mean(),
        "Std_Level": s.std(),
    })

overview = pd.DataFrame(overview_rows)

display(overview)

overview.to_csv(
    TABLE_DIR / "03_development_overview.csv",
    index=False
)

# 4. Visual inspection of TT levels

In [ ]:
for cur in CURRENCIES:
    plt.figure(figsize=(12, 4))
    plt.plot(df["Date"], df[f"{cur}_mid"], linewidth=1)

    plt.title(f"{cur}/LKR TT Mid — Development Period")
    plt.xlabel("Date")
    plt.ylabel("TT Mid")
    plt.tight_layout()

    plt.savefig(
        FIGURE_DIR / f"03_{cur}_development_level.png",
        dpi=150,
        bbox_inches="tight"
    )

    plt.show()

# 5. Return and change construction

We inspect:

\[
\Delta P_t = P_t - P_{t-1}
\]

and

\[
r_t = \ln(P_t/P_{t-1})
\]

These are diagnostic series only.

In [ ]:
for cur in CURRENCIES:
    mid = pd.to_numeric(df[f"{cur}_mid"], errors="coerce")

    df[f"{cur}_diff1_diag"] = mid.diff(1)
    df[f"{cur}_log_return_diag"] = np.log(mid / mid.shift(1))
    df[f"{cur}_abs_return_diag"] = df[f"{cur}_log_return_diag"].abs()

print("Diagnostic change/return series created.")

# 6. Stationarity tests

We use two complementary tests:

### ADF
Null hypothesis: **unit root / non-stationary**

A small p-value supports stationarity.

### KPSS
Null hypothesis: **stationary**

A small p-value suggests non-stationarity.

Using both together gives a clearer interpretation.

In [ ]:
# ============================================================
# 6. ADF + KPSS HELPERS
# ============================================================

def run_adf(series):
    s = pd.Series(series).dropna()

    if len(s) < 30:
        return np.nan, np.nan

    result = adfuller(
        s,
        autolag="AIC"
    )

    return float(result[0]), float(result[1])

def run_kpss(series):
    s = pd.Series(series).dropna()

    if len(s) < 30:
        return np.nan, np.nan

    try:
        result = kpss(
            s,
            regression="c",
            nlags="auto"
        )

        return float(result[0]), float(result[1])

    except Exception:
        return np.nan, np.nan

In [ ]:
stationarity_rows = []

for cur in CURRENCIES:
    candidates = {
        "Level": df[f"{cur}_mid"],
        "First_Difference": df[f"{cur}_diff1_diag"],
        "Log_Return": df[f"{cur}_log_return_diag"],
    }

    for name, series in candidates.items():
        adf_stat, adf_p = run_adf(series)
        kpss_stat, kpss_p = run_kpss(series)

        if np.isfinite(adf_p) and np.isfinite(kpss_p):
            if adf_p < 0.05 and kpss_p >= 0.05:
                interpretation = "Stationary"
            elif adf_p >= 0.05 and kpss_p < 0.05:
                interpretation = "Non-stationary"
            else:
                interpretation = "Mixed / borderline"
        else:
            interpretation = "Insufficient / test issue"

        stationarity_rows.append({
            "Currency": cur,
            "Series": name,
            "ADF_Stat": adf_stat,
            "ADF_p": adf_p,
            "KPSS_Stat": kpss_stat,
            "KPSS_p": kpss_p,
            "Interpretation": interpretation,
        })

stationarity = pd.DataFrame(stationarity_rows)

display(stationarity)

stationarity.to_csv(
    TABLE_DIR / "03_stationarity_tests.csv",
    index=False
)

# 7. Forecast-target stationarity

For H1, H3 and H7, we compare:

- future level;
- future absolute change;
- future log change.

This does **not** choose the final target using holdout performance.  
It only tells us which formulations are statistically more stable.

In [ ]:
target_stationarity_rows = []

for cur in CURRENCIES:
    for h in HORIZONS:
        candidates = {
            "Future_Level": df[f"{cur}_target_H{h}"],
            "Future_Change": df[f"{cur}_change_H{h}"],
            "Future_Log_Change": df[f"{cur}_log_change_H{h}"],
        }

        for name, series in candidates.items():
            adf_stat, adf_p = run_adf(series)
            kpss_stat, kpss_p = run_kpss(series)

            if np.isfinite(adf_p) and np.isfinite(kpss_p):
                if adf_p < 0.05 and kpss_p >= 0.05:
                    interpretation = "Stationary"
                elif adf_p >= 0.05 and kpss_p < 0.05:
                    interpretation = "Non-stationary"
                else:
                    interpretation = "Mixed / borderline"
            else:
                interpretation = "Insufficient / test issue"

            target_stationarity_rows.append({
                "Currency": cur,
                "Horizon": h,
                "Target_Form": name,
                "ADF_p": adf_p,
                "KPSS_p": kpss_p,
                "Interpretation": interpretation,
            })

target_stationarity = pd.DataFrame(
    target_stationarity_rows
)

display(target_stationarity)

target_stationarity.to_csv(
    TABLE_DIR / "03_target_stationarity.csv",
    index=False
)

# 8. ACF / PACF diagnostics

We inspect up to 30 lags for both:

- TT level
- TT log return

The figures are descriptive and help justify later time-series specifications.

In [ ]:
# ============================================================
# 8. ACF / PACF PLOTS
# ============================================================

MAX_LAG = 30

for cur in CURRENCIES:
    for series_name, series in [
        ("Level", df[f"{cur}_mid"]),
        ("Log_Return", df[f"{cur}_log_return_diag"]),
    ]:
        s = pd.Series(series).dropna()

        acf_vals = acf(
            s,
            nlags=MAX_LAG,
            fft=True
        )

        pacf_vals = pacf(
            s,
            nlags=MAX_LAG,
            method="ywm"
        )

        plt.figure(figsize=(10, 4))
        plt.stem(
            range(len(acf_vals)),
            acf_vals
        )
        plt.axhline(0)
        plt.title(f"{cur} {series_name} — ACF")
        plt.xlabel("Lag")
        plt.ylabel("ACF")
        plt.tight_layout()

        plt.savefig(
            FIGURE_DIR / f"03_{cur}_{series_name}_acf.png",
            dpi=150,
            bbox_inches="tight"
        )
        plt.show()

        plt.figure(figsize=(10, 4))
        plt.stem(
            range(len(pacf_vals)),
            pacf_vals
        )
        plt.axhline(0)
        plt.title(f"{cur} {series_name} — PACF")
        plt.xlabel("Lag")
        plt.ylabel("PACF")
        plt.tight_layout()

        plt.savefig(
            FIGURE_DIR / f"03_{cur}_{series_name}_pacf.png",
            dpi=150,
            bbox_inches="tight"
        )
        plt.show()

# 9. Ljung–Box serial-correlation test

Null hypothesis:

> no autocorrelation up to the tested lag.

We test log returns at lags 5, 10 and 20.

In [ ]:
ljung_rows = []

for cur in CURRENCIES:
    s = df[f"{cur}_log_return_diag"].dropna()

    lb = acorr_ljungbox(
        s,
        lags=[5, 10, 20],
        return_df=True
    )

    for lag, row in lb.iterrows():
        ljung_rows.append({
            "Currency": cur,
            "Lag": int(lag),
            "LB_Stat": float(row["lb_stat"]),
            "LB_p": float(row["lb_pvalue"]),
            "Serial_Dependence": (
                "Yes"
                if row["lb_pvalue"] < 0.05
                else "No"
            )
        })

ljung_box = pd.DataFrame(ljung_rows)

display(ljung_box)

ljung_box.to_csv(
    TABLE_DIR / "03_ljung_box_returns.csv",
    index=False
)

# 10. ARCH-LM volatility-clustering test

Null hypothesis:

> no ARCH effect.

A significant result supports modelling conditional volatility separately.

In [ ]:
arch_rows = []

for cur in CURRENCIES:
    s = df[f"{cur}_log_return_diag"].dropna()

    lm_stat, lm_p, f_stat, f_p = het_arch(
        s,
        nlags=10
    )

    arch_rows.append({
        "Currency": cur,
        "ARCH_LM_Stat": float(lm_stat),
        "ARCH_LM_p": float(lm_p),
        "ARCH_Effect": (
            "Yes"
            if lm_p < 0.05
            else "No"
        )
    })

arch_test = pd.DataFrame(arch_rows)

display(arch_test)

arch_test.to_csv(
    TABLE_DIR / "03_arch_lm_test.csv",
    index=False
)

# 11. Rolling volatility

This is a descriptive view of changing market risk.

We use a 30-observation rolling standard deviation of log returns.

In [ ]:
for cur in CURRENCIES:
    r = df[f"{cur}_log_return_diag"]

    rolling_vol = (
        r.rolling(30, min_periods=15).std()
    )

    plt.figure(figsize=(12, 4))
    plt.plot(
        df["Date"],
        rolling_vol,
        linewidth=1
    )

    plt.title(
        f"{cur}/LKR — 30-Observation Rolling Volatility"
    )
    plt.xlabel("Date")
    plt.ylabel("Std. dev. of log return")
    plt.tight_layout()

    plt.savefig(
        FIGURE_DIR / f"03_{cur}_rolling_volatility.png",
        dpi=150,
        bbox_inches="tight"
    )

    plt.show()

# 12. Simple volatility regimes

For descriptive robustness analysis only, we classify each currency's rolling volatility into:

- Low
- Medium
- High

using development-period terciles.

These regimes are **not used as future-informed labels** in later model fitting.

In [ ]:
regime_rows = []

for cur in CURRENCIES:
    vol = (
        df[f"{cur}_log_return_diag"]
        .rolling(30, min_periods=15)
        .std()
    )

    q1 = vol.quantile(1/3)
    q2 = vol.quantile(2/3)

    regime = pd.Series(
        np.where(
            vol <= q1,
            "Low",
            np.where(
                vol <= q2,
                "Medium",
                "High"
            )
        ),
        index=df.index
    )

    regime = regime.where(vol.notna())

    df[f"{cur}_volatility_regime_diag"] = regime

    counts = (
        regime.value_counts(dropna=True)
        .reindex(["Low", "Medium", "High"])
        .fillna(0)
    )

    for r_name, count in counts.items():
        regime_rows.append({
            "Currency": cur,
            "Regime": r_name,
            "Rows": int(count)
        })

regime_summary = pd.DataFrame(regime_rows)

display(regime_summary)

regime_summary.to_csv(
    TABLE_DIR / "03_volatility_regime_summary.csv",
    index=False
)

# 13. Rolling cross-currency correlations

We inspect 60-observation rolling correlations of daily log returns.

This helps show whether relationships among USD, GBP, EUR and JPY remain stable over time.

In [ ]:
pairs = [
    ("USD", "GBP"),
    ("USD", "EUR"),
    ("USD", "JPY"),
    ("GBP", "EUR"),
    ("GBP", "JPY"),
    ("EUR", "JPY"),
]

rolling_corr_rows = []

for a, b in pairs:
    corr = (
        df[f"{a}_log_return_diag"]
        .rolling(60, min_periods=30)
        .corr(df[f"{b}_log_return_diag"])
    )

    df[f"corr60_{a}_{b}"] = corr

    rolling_corr_rows.append({
        "Pair": f"{a}-{b}",
        "Mean_Rolling_Correlation": corr.mean(),
        "Min_Rolling_Correlation": corr.min(),
        "Max_Rolling_Correlation": corr.max(),
        "Std_Rolling_Correlation": corr.std(),
    })

rolling_corr_summary = pd.DataFrame(
    rolling_corr_rows
)

display(rolling_corr_summary)

rolling_corr_summary.to_csv(
    TABLE_DIR / "03_rolling_correlation_summary.csv",
    index=False
)

In [ ]:
plt.figure(figsize=(12, 6))

for a, b in pairs:
    plt.plot(
        df["Date"],
        df[f"corr60_{a}_{b}"],
        linewidth=1,
        label=f"{a}-{b}"
    )

plt.title("60-Observation Rolling Correlations of TT Log Returns")
plt.xlabel("Date")
plt.ylabel("Correlation")
plt.legend(ncol=2)
plt.tight_layout()

plt.savefig(
    FIGURE_DIR / "03_rolling_cross_currency_correlations.png",
    dpi=150,
    bbox_inches="tight"
)

plt.show()

# 14. Structural-break analysis

We use the Zivot–Andrews unit-root test, which allows one endogenous structural break.

The detected break date is descriptive evidence of major instability.

We do not manually move observations between training and testing because of the detected date.

In [ ]:
za_rows = []

for cur in CURRENCIES:
    s_full = df[["Date", f"{cur}_mid"]].dropna().copy()

    s = s_full[f"{cur}_mid"].to_numpy()

    try:
        za_stat, za_p, za_crit, usedlag, break_index = zivot_andrews(
            s,
            maxlag=12,
            regression="ct",
            autolag="AIC"
        )

        break_index = int(break_index)

        if 0 <= break_index < len(s_full):
            break_date = s_full.iloc[break_index]["Date"]
        else:
            break_date = pd.NaT

        za_rows.append({
            "Currency": cur,
            "ZA_Stat": float(za_stat),
            "ZA_p": float(za_p),
            "Break_Index": break_index,
            "Break_Date": break_date,
            "Used_Lag": int(usedlag),
        })

    except Exception as e:
        za_rows.append({
            "Currency": cur,
            "ZA_Stat": np.nan,
            "ZA_p": np.nan,
            "Break_Index": np.nan,
            "Break_Date": pd.NaT,
            "Used_Lag": np.nan,
        })

zivot_results = pd.DataFrame(za_rows)

display(zivot_results)

zivot_results.to_csv(
    TABLE_DIR / "03_zivot_andrews_breaks.csv",
    index=False
)

# 15. CUSUM stability check

We also test whether a simple time-trend regression remains structurally stable.

This is another descriptive stability diagnostic.

In [ ]:
cusum_rows = []

for cur in CURRENCIES:
    temp = df[["Date", f"{cur}_mid"]].dropna().copy()

    y = temp[f"{cur}_mid"].to_numpy()
    t = np.arange(len(y))

    X = sm.add_constant(t)

    model = sm.OLS(y, X).fit()

    stat, p_value, crit = breaks_cusumolsresid(
        model.resid,
        ddof=int(model.df_model)
    )

    cusum_rows.append({
        "Currency": cur,
        "CUSUM_Stat": float(stat),
        "CUSUM_p": float(p_value),
        "Stable_at_5pct": (
            "Yes"
            if p_value >= 0.05
            else "No"
        )
    })

cusum_results = pd.DataFrame(cusum_rows)

display(cusum_results)

cusum_results.to_csv(
    TABLE_DIR / "03_cusum_stability.csv",
    index=False
)

# 16. Year-by-year return behaviour

This gives a compact regime-oriented summary without overcomplicating the analysis.

In [ ]:
yearly_return_rows = []

for cur in CURRENCIES:
    temp = df[
        ["Date", f"{cur}_log_return_diag"]
    ].copy()

    temp["Year"] = temp["Date"].dt.year

    for year, g in temp.groupby("Year"):
        r = g[f"{cur}_log_return_diag"].dropna()

        if len(r) == 0:
            continue

        yearly_return_rows.append({
            "Currency": cur,
            "Year": int(year),
            "Observations": len(r),
            "Mean_Return": r.mean(),
            "Return_Std": r.std(),
            "Mean_Absolute_Return": r.abs().mean(),
            "Min_Return": r.min(),
            "Max_Return": r.max(),
        })

yearly_return_summary = pd.DataFrame(
    yearly_return_rows
)

display(yearly_return_summary)

yearly_return_summary.to_csv(
    TABLE_DIR / "03_yearly_return_summary.csv",
    index=False
)

# 17. Compact modelling implications

This section translates diagnostics into modelling guidance.

It does **not** choose a winning model.

In [ ]:
implication_rows = []

for cur in CURRENCIES:
    level_row = stationarity[
        (stationarity["Currency"] == cur)
        & (stationarity["Series"] == "Level")
    ].iloc[0]

    return_row = stationarity[
        (stationarity["Currency"] == cur)
        & (stationarity["Series"] == "Log_Return")
    ].iloc[0]

    lb_row = ljung_box[
        (ljung_box["Currency"] == cur)
        & (ljung_box["Lag"] == 10)
    ].iloc[0]

    arch_row = arch_test[
        arch_test["Currency"] == cur
    ].iloc[0]

    cusum_row = cusum_results[
        cusum_results["Currency"] == cur
    ].iloc[0]

    implication_rows.append({
        "Currency": cur,
        "Level_Stationarity": level_row["Interpretation"],
        "Return_Stationarity": return_row["Interpretation"],
        "Return_Autocorrelation_Lag10": lb_row["Serial_Dependence"],
        "ARCH_Effect": arch_row["ARCH_Effect"],
        "Structural_Stability": cusum_row["Stable_at_5pct"],
        "Implication": (
            "Use differenced/change-based statistical formulation; "
            "retain volatility branch; evaluate models across rolling years/regimes."
        )
    })

modelling_implications = pd.DataFrame(
    implication_rows
)

display(modelling_implications)

modelling_implications.to_csv(
    TABLE_DIR / "03_modelling_implications.csv",
    index=False
)

# 18. Quality gate

In [ ]:
quality_gate = pd.DataFrame([
    {
        "Area": "Final holdout isolation",
        "Status": "PASS",
        "Evidence": "Only 02_development_model_dataset.csv loaded"
    },
    {
        "Area": "Stationarity",
        "Status": "PASS",
        "Evidence": "ADF and KPSS applied to levels, differences and returns"
    },
    {
        "Area": "Target formulation",
        "Status": "PASS",
        "Evidence": "Future level/change/log-change diagnostics completed"
    },
    {
        "Area": "Serial dependence",
        "Status": "PASS",
        "Evidence": "Ljung-Box test on returns at lags 5, 10 and 20"
    },
    {
        "Area": "Volatility clustering",
        "Status": "PASS",
        "Evidence": "ARCH-LM test completed"
    },
    {
        "Area": "Structural breaks",
        "Status": "PASS",
        "Evidence": "Zivot-Andrews and CUSUM diagnostics completed"
    },
    {
        "Area": "Cross-currency dynamics",
        "Status": "PASS",
        "Evidence": "Rolling return correlations analysed"
    },
    {
        "Area": "Regime diagnostics",
        "Status": "PASS",
        "Evidence": "Low/medium/high volatility regimes summarized"
    },
])

display(quality_gate)

quality_gate.to_csv(
    TABLE_DIR / "03_quality_gate.csv",
    index=False
)

# 19. Final summary

After this notebook, we should know whether the data support:

- first-differenced / change-based statistical models;
- machine-learning models using stable target transformations;
- a separate volatility branch;
- rolling/regime-aware evaluation.

The next notebook should establish the **Random Walk benchmark**.

In [ ]:
print("=" * 76)
print("NOTEBOOK 03 — EDA, DIAGNOSTICS AND REGIME ANALYSIS COMPLETE")
print("=" * 76)

print(
    "Development period:",
    df["Date"].min().date(),
    "to",
    df["Date"].max().date()
)

print("\nStationarity summary:")
display(stationarity)

print("\nARCH summary:")
display(arch_test)

print("\nStructural-break summary:")
display(zivot_results)

print("\nModelling implications:")
display(modelling_implications)

print("\nIMPORTANT:")
print("No 2024-2025 holdout observations were evaluated.")
print("Next notebook: 04_Random_Walk_Baseline.ipynb")